# SimCLR implementation

In [11]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torchvision
import torchvision.models as models

from torch.utils.data import DataLoader
from tqdm import tqdm

### Augmentation Transform

In [12]:
class SimCLRTransform:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=32),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)

### ResNet-18 Encoder

In [13]:

class ResNet18Encoder(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18Encoder, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()  # Remove maxpool layer for CIFAR-10
        self.resnet.fc = nn.Identity()  # Remove fully connected layer for feature extraction

    def forward(self, x):
        return self.resnet(x)

### Projection Head

In [14]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### NT_Xent Loss

In [15]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)  # Concatenate positive pairs
    
    z = nn.functional.normalize(z, dim=1)  # L2 Norm
    similarity_matrix = torch.matmul(z, z.T)/temperature  # Compute similarity matrix

    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    similarity_matrix = similarity_matrix[~mask].view(2 * batch_size, -1)  # Remove self-similarity

    targets = torch.cat([
        torch.arange(batch_size -1, 2*batch_size -1),
        torch.arange(0, batch_size)
    ]).to(z.device)

    loss = nn.CrossEntropyLoss()(similarity_matrix, targets)
    return loss

### Import Dataset

In [16]:
simclr_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                                download=False, transform=SimCLRTransform())
simclr_loader = DataLoader(simclr_trainset, batch_size=256, shuffle=True, num_workers=2, drop_last=True)


### Training Loop Setup

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Initialize models
encoder = ResNet18Encoder().to(device)
projection_head = ProjectionHead().to(device)

# Optimizer
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(projection_head.parameters()), lr=3e-4)

# Training
num_epochs = 200

Using device: cuda


### Training Loop

In [19]:
for epoch in range(num_epochs):                                                                                                     
    running_loss = 0.0                                                                                                              
    loop = tqdm(simclr_loader, desc=f'Epoch [{epoch+1}/{num_epochs}]', leave=False)                                               
    for (view1, view2), _ in loop:                                                                                                  
        view1, view2 = view1.to(device), view2.to(device)                                                                           
                                                                                                                                    
        h_i = encoder(view1)                                                                                                        
        h_j = encoder(view2)                                                                                                        
        z_i = projection_head(h_i)
        z_j = projection_head(h_j)

        loss = nt_xent_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(simclr_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Avg Loss: {avg_loss:.4f}')

# Save encoder weights
torch.save(encoder.state_dict(), 'simclr_encoder_200ep.pth')
print('Training complete. Encoder saved.')

Epoch [1/200], Avg Loss: 5.2183


Epoch [2/200], Avg Loss: 5.1332


Epoch [3/200], Avg Loss: 5.0723


Epoch [4/200], Avg Loss: 5.0247


Epoch [5/200], Avg Loss: 4.9934


...

Epoch [196/200], Avg Loss: 4.6572


Epoch [197/200], Avg Loss: 4.6588


Epoch [198/200], Avg Loss: 4.6559


Epoch [199/200], Avg Loss: 4.6549


Epoch [200/200], Avg Loss: 4.6554
Training complete. Encoder saved.
